In [1]:
import numpy as np
import gmsh

In [2]:
x = [0, 200, 800, 900]
y = [-15, -15, 5, 5]
water_level = 0
dz = 0.1               # thickness for 2D
air_height = 20
mesh_res = [200, 500, 50]  # horizontal divisions per segment
cells_y_water = 20
cells_y_air = 10

In [4]:

x = [0, 200, 800, 900]
y = [-15, -15, 5, 5]
water_level = 0
air_height = 20
dz = 0.1  # thickness in Z (for 2D block)
mesh_res = [200, 500, 50]   # horizontal divisions
cells_y_water = 20
cells_y_air = 10

# ---------------------------
# INITIALIZE GMSH
# ---------------------------
gmsh.initialize()
gmsh.model.add("wave_slope")

# ---------------------------
# CREATE POINTS
# ---------------------------
pts = []  # store point tags

# function to add point
def add_pt(xx, yy, zz=0):
    return gmsh.model.occ.addPoint(xx, yy, zz)

# bottom/water/air points for front and back
for z in [0, dz]:
    for i in range(len(x)):
        pts.append(add_pt(x[i], y[i], z))                # bottom
        pts.append(add_pt(x[i], water_level, z))         # water surface
        pts.append(add_pt(x[i], water_level + air_height, z))  # top air

# ---------------------------
# CREATE LINES
# ---------------------------
lines = []
npts = len(x)*3  # points per z-layer
for zlayer in range(2):  # front/back
    base = zlayer * npts
    for i in range(len(x)-1):
        # bottom segment
        lines.append(gmsh.model.occ.addLine(pts[base + 3*i + 0],
                                            pts[base + 3*(i+1) + 0]))
        # water surface segment
        lines.append(gmsh.model.occ.addLine(pts[base + 3*i + 1],
                                            pts[base + 3*(i+1) + 1]))
        # top air segment
        lines.append(gmsh.model.occ.addLine(pts[base + 3*i + 2],
                                            pts[base + 3*(i+1) + 2]))

    # vertical lines connecting layers
    for i in range(len(x)):
        lines.append(gmsh.model.occ.addLine(pts[base + 3*i + 0],
                                            pts[base + 3*i + 1]))
        lines.append(gmsh.model.occ.addLine(pts[base + 3*i + 1],
                                            pts[base + 3*i + 2]))

# ---------------------------
# CREATE SURFACES
# ---------------------------
# loop through each horizontal segment and build 2 surfaces (water and air)
surfaces = []
for zlayer in range(2):
    base = zlayer * npts
    for i in range(len(x)-1):
        # water region surface
        c1 = [pts[base + 3*i + 0], pts[base + 3*(i+1) + 0],
              pts[base + 3*(i+1) + 1], pts[base + 3*i + 1]]
        l1 = [gmsh.model.occ.addLine(c1[0], c1[1]),
              gmsh.model.occ.addLine(c1[1], c1[2]),
              gmsh.model.occ.addLine(c1[2], c1[3]),
              gmsh.model.occ.addLine(c1[3], c1[0])]
        loop1 = gmsh.model.occ.addCurveLoop(l1)
        s1 = gmsh.model.occ.addPlaneSurface([loop1])

        # air region surface
        c2 = [pts[base + 3*i + 1], pts[base + 3*(i+1) + 1],
              pts[base + 3*(i+1) + 2], pts[base + 3*i + 2]]
        l2 = [gmsh.model.occ.addLine(c2[0], c2[1]),
              gmsh.model.occ.addLine(c2[1], c2[2]),
              gmsh.model.occ.addLine(c2[2], c2[3]),
              gmsh.model.occ.addLine(c2[3], c2[0])]
        loop2 = gmsh.model.occ.addCurveLoop(l2)
        s2 = gmsh.model.occ.addPlaneSurface([loop2])

        surfaces.append((s1, i))  # (surface tag, block index)
        surfaces.append((s2, i))

# sync to build the CAD model
gmsh.model.occ.synchronize()

# ---------------------------
# TRANFINITE MESH (STRUCTURED)
# ---------------------------
for srf, idx in surfaces:
    # horizontal spans
    nx = mesh_res[idx]
    ny = cells_y_water if srf % 2 == 0 else cells_y_air
    gmsh.model.mesh.setTransfiniteSurface(srf, "Left")

    # transfinite curves
    gmsh.model.mesh.setTransfiniteCurve(surfaces[idx*2][0], nx+1)
    gmsh.model.mesh.setTransfiniteCurve(surfaces[idx*2+1][0], nx+1)

# set transfinite surfaces into structured quads
# sync CAD model so surfaces exist in the mesh
gmsh.model.occ.synchronize()

# recombine all surfaces properly
for srf, idx in surfaces:
    gmsh.model.mesh.setRecombine(2, srf)
# ---------------------------
# PHYSICAL GROUPS (BOUNDARY TAGS)
# ---------------------------
# inlet group (left side)
# outlet group (right side)
inlet_lines = []
outlet_lines = []
bottom_lines = []
top_lines = []

for i in range(len(x)):
    # vertical lines on the left (inlet)
    if i == 0:
        l = gmsh.model.occ.addLine(pts[i], pts[i + n_points_per_layer])
        inlet_lines.append(l)
    # vertical lines on the right (outlet)
    if i == len(x)-1:
        l = gmsh.model.occ.addLine(pts[i], pts[i + n_points_per_layer])
        outlet_lines.append(l)
idx_out = len(lines) - 3

gmsh.model.addPhysicalGroup(1, inlet_lines, name="inlet")
gmsh.model.addPhysicalGroup(1, outlet_lines, name="outlet")

# bottom group
bot_lines = [l for l in lines if "bottom" in str(l)]
gmsh.model.addPhysicalGroup(1, bot_lines, name="bottom")
# top group
top_lines = [l for l in lines if "air" in str(l)]
gmsh.model.addPhysicalGroup(1, top_lines, name="top")
# front/back
# TODO: create groups for front/back if needed

# ---------------------------
# GENERATE MESH
# ---------------------------
gmsh.model.mesh.generate(2)
gmsh.write("mesh.msh")

print("✔ Mesh generated!")

gmsh.finalize()

NameError: name 'n_points_per_layer' is not defined